In [1]:
DATA_PATH = (
  "hest/bench_data/CCRCC",
  "hest/bench_data/PRAD",
  "hest/kidney",
  "hest/andrew",
  "hest/andersson",
  # "nameeta/GBM",
  "takano/xenium",
  "takano/visium"
)

data_dir = '/home/shared/chungym/hier_st/input'


In [2]:
import os
from loki.utils import load_model, encode_images_from_h5, encode_text_df
from loki.preprocess import generate_gene_df

# model_path = os.path.join('../', 'checkpoint.pt')
model_path = '/home/chungym/.cache/huggingface/hub/models--WangGuangyuLab--Loki/snapshots/8e08d3a131428feed2341167084d7b15c143a2a7/checkpoint.pt'
device = 'cuda'
model, preprocess, tokenizer = load_model(model_path, device)

/home/chungym/miniconda3/envs/TRIPLEX/lib/python3.11/site-packages/open_clip/factory.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint

In [3]:
from typing import Union
import scanpy as sc
import pandas as pd
from scipy.sparse import issparse
import torch


def encode_text_one_slide(
    model: torch.nn.Module,
    tokenizer: callable,
    st_path: str,
    device: Union[str, torch.device]):
    
    ad = sc.read_h5ad(st_path)
    house_keeping_genes = pd.read_csv('housekeeping_genes.csv', index_col=0)
    top_k_genes_str = generate_gene_df(ad, house_keeping_genes, todense=issparse(ad.X))
    text_embeddings = encode_text_df(model, tokenizer, top_k_genes_str, 'label', device)
    return text_embeddings


In [15]:
import os
from tqdm import tqdm
import pandas as pd
import scanpy as sc
import torch

for data in DATA_PATH:

    data_path = os.path.join(data_dir, data)
    split_path = os.path.join(data_path, 'splits')
    
    num_folds = len(os.listdir(split_path)) // 2
    for fold in range(num_folds):
        
        print(f"Processing dataset: {data}, fold: {fold}")
        query_path = os.path.join(data_path, 'splits', f'test_{fold}.csv')
        query_samples = pd.read_csv(query_path).sample_id.tolist()
        
        key_path = os.path.join(data_path, 'splits', f'train_{fold}.csv')
        key_samples = pd.read_csv(key_path).sample_id.tolist()

    
        print("Encoding text of key samples...")
        if fold == 0:
            key_st_embeddings_dict = {sample: encode_text_one_slide(
                                                model, tokenizer,
                                                f"{data_path}/adata/{sample}.h5ad", device).detach().cpu() \
                                                for sample in tqdm(key_samples)}
            
        else:
            key_st_embeddings_dict.update({sample: encode_text_one_slide(
                                                model, tokenizer,
                                                f"{data_path}/adata/{sample}.h5ad", device).detach().cpu() \
                                                for sample in tqdm(key_samples) if sample not in key_st_embeddings_dict})
        key_st_embeddings = [key_st_embeddings_dict[sample] for sample in key_samples]
        
        key_st_embeddings = torch.cat(key_st_embeddings, dim=0)
        
        print("Encoding images of query samples...")
        query_img_embeddings = [encode_images_from_h5(
            model, preprocess,
            f"{data_path}/patches/{sample}.h5", device).detach().cpu() \
            for sample in tqdm(query_samples)]
        query_img_embeddings = torch.cat(query_img_embeddings, dim=0)

        dot_similarity = query_img_embeddings @ key_st_embeddings.T
        

Processing dataset: hest/bench_data/CCRCC, fold: 0


In [11]:
query_samples

['LUAD_No_16',
 'LUAD_No_2',
 'FFPE_LUAD_No2_A',
 'FFPE_LUAD_No2_B',
 'FFPE_LUAD_No2_D',
 'TSU-22',
 'TSU-32',
 'TSU-35',
 'TSU-41']

In [11]:
import h5py

# root_dir = '/home/shared/chungym/hier_st/input/'
data_dir='/home/shared/chungym/hier_st/input/takano/xenium/patches'
file_name = 'Xenium_LUAD_No14.h5'
with h5py.File(os.path.join(data_dir, file_name), 'r') as f:
    print(len(f['img']))
# file_name='Xenium_LUAD_No14.h5ad'




5138


In [ ]:
from loki.utils import encode_images_from_h5

data_dir='/home/shared/chungym/hier_st/input/takano/xenium/patches'
file_name='Xenium_LUAD_No14.h5'
image_path = os.path.join(data_dir, file_name)

image_embeddings = encode_images_from_h5(model, preprocess, image_path, device)



In [12]:
image_embeddings.shape

torch.Size([5138, 768])

AnnData object with n_obs × n_vars = 5138 × 541
    obs: 'in_tissue', 'pxl_col_in_fullres', 'pxl_row_in_fullres', 'array_col', 'array_row', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'spatial'
    obsm: 'spatial'

,label
003x011,COL1A1 KRT8 SPARC TNC HMGA1 SDC4 WSB1 ERBB2 CD...
004x008,SPARC KRT8 COL1A1 ACTA2 TNC ERBB2 POSTN HMGA1 ...
004x009,SPARC PTCRA COL1A1 ACTA2 POSTN WSB1 PECAM1 SPP...
004x010,PTCRA SPARC COL1A1 POSTN CD44 PTPRC HMGA1 ACTA...
004x011,COL1A1 SPARC PTCRA POSTN HMGA1 ACTA2 CTSK MMP1...
...,...
098x068,SFTPB SDC4 SFTPC WSB1 MARCO MSR1 FABP4 CD44 CD...
098x069,SFTPB SFTPC WSB1 SDC4 ICAM1 CD44 NAPSA SCGB3A2...
098x070,MARCO MSR1 PTCRA SFTPB FABP4 WSB1 APOE CD68 CD...
098x071,PTCRA SPARC ICAM1 VEGFA CD44 POSTN COL1A1 WSB1...


In [6]:
from loki.utils import encode_text_df

text_embeddings = encode_text_df(model, tokenizer, top_k_genes_str, 'label', device)

In [7]:
text_embeddings.shape

torch.Size([5138, 768])

In [ ]:
dot_similarity = image_embeddings @ text_embeddings.T
